데이터 불러오기

In [13]:
import pandas as pd
from pathlib import Path

welcome1 = pd.read_excel("./Welcome/Welcome_1st.xlsx", sheet_name=None)
welcome2 = pd.read_excel("./Welcome/Welcome_2nd.xlsx", sheet_name=None)

qpoll = []
for f in sorted(Path("Quickpoll").glob("qpoll_join_*.xlsx")):
    sheets = pd.read_excel(f, sheet_name=None)
    qpoll.append(sheets)


컬럼명 전처리

In [14]:
import re

counter = 0

for df in [welcome1['data'], welcome2['data']]:
    new_cols = []
    pre_col = ''

    for idx, col in enumerate(df.columns):
        if re.search(r'Q\d+', str(col)):
            suffix_m = re.search(r'(_\d+)$', str(col))
            suffix = suffix_m.group(1) if suffix_m else ''
            base = re.sub(r'(_\d+)$', '', str(col))

            if not re.search(r'Q\d+', str(pre_col)) or re.search(r'Q\d+', str(pre_col)).group(0) != re.search(r'Q\d+', str(col)).group(0):
                counter += 1

            new_base = re.sub(r'Q\d+', f'Q{counter}', base, count=1)
            new_cols.append(f"{new_base}{suffix}")
        else:
            new_cols.append(col)
        pre_col = col
    df.columns = new_cols

for idx, qpoll_item in enumerate(qpoll):
    for key, value in qpoll_item.items():
        if key.startswith('qpoll_join_'):
            qpoll_item[key] = pd.DataFrame(value.values[1:], columns=value.iloc[0])
            qpoll_item[key].rename(columns={'구분':'Carrier', '고유번호':'mb_sn', '성별':'Sex', '나이':'Birth', '지역':'Region'}, inplace=True)
            for label in qpoll_item[key].columns:
                if re.search(r'문항\d+', str(label)):
                    counter+= 1
                    new_label = re.sub(r'문항\d+', f'Q{counter}', str(label), count=1)
                    qpoll_item[key].rename(columns={label: new_label}, inplace=True)
        else:
            qpoll_item[key].loc[-1] = value.columns
            qpoll_item[key].index = value.index + 1


코드북 생성

In [15]:
codebook = pd.DataFrame(columns=['variable', 'question_text', 'type', 'options_json'])
codebook = pd.concat([
    codebook,
    pd.DataFrame([{'variable': 'Q0', 'question_text': '통신사', 'type': 'STRING', 'options_json': None}])
], ignore_index=True)

rows = []
pre_Q = None
options_json = None

for label in [welcome1['lable'], welcome2['label']]:
    for row in label.itertuples():
        if row.변수명 == 'mb_sn':
            continue

        num = re.search(r'Q(\d+)', str(row.변수명))

        if num:
            adjustment = -9 if label is welcome1['lable'] else 3
            new_variable = re.sub(r'Q\d+', f'Q{int(num.group(1)) + adjustment}', row.변수명, count=1)

            if options_json is not None and pre_Q is not None:
                rows[-1]['options_json'] = options_json
                options_json = None

            rows.append(
                {
                    'variable' : new_variable,
                    'question_text' : row.문항,
                    'type' : row.문항유형,
                    'options_json' : None,
                })
            pre_Q = row.변수명

        else:
            if options_json is None:
                options_json = {}
            options_json[str(row.변수명)] = row.문항

if options_json is not None and pre_Q is not None:
    rows[-1]['options_json'] = options_json
    options_json = None

codebook = pd.concat([codebook, pd.DataFrame(rows)], ignore_index=True)

rows = []
options_json = None
counter = 16
for idx, qpoll_item in enumerate(qpoll):
    type = []
    for key, value in qpoll_item.items():
        
        if key.startswith('qpoll_join_'):
            for label in value.columns:
                if re.search(r'Q\d+', str(label)):
                    type.append('MULTI' if any(
                        isinstance(ans, list) or ',' in str(ans) for ans in value[label]
                        ) else 'SINGLE')

        else:
            for i in range(0, len(value), 2):
                row = value.iloc[i:i+2]
                if len(row) == 2:
                    combined_df = pd.DataFrame(row)
                    question_text = None
                    options_json = {}

                    for label in combined_df.columns:
                        if label == '설문제목':
                            counter += 1
                            question_text = combined_df[label].iloc[0]
                            rows.append(
                                {
                                    'variable': 'Q' + str(counter),
                                    'question_text': question_text,
                                    'type': type[i // 2] if i // 2 < len(type) else 'UNKNOWN',
                                    'options_json': None,
                                }
                            )
                        num = re.search(r'보기(\d+)$', str(label))
                        if num:
                            opt_text = combined_df[label].iloc[0]
                            options_json[str(num.group(1))] = opt_text

                    if options_json and len(rows) > 0:
                        rows[-1]['options_json'] = options_json

codebook = pd.concat([codebook, pd.DataFrame(rows)], ignore_index=True)
codebook['type'] = codebook['type'].str.upper()

# Adjust specific variable types
codebook.loc[codebook['variable'] == 'Q3_1', 'variable'] = 'Q3_2'
codebook.loc[codebook['variable'] == 'Q3', 'variable'] = 'Q3_1'
codebook.loc[codebook['variable'] == 'Q2', 'type'] = 'STRING'

codebook.to_csv('codebook.csv', index=False, encoding='utf-8-sig')

설문 데이터 병합

In [51]:
# 중복된 열을 유지한 채 병합
data = None
for idx, qpoll_item in enumerate(qpoll):
    for key, value in qpoll_item.items():
        if key.startswith('qpoll_join_'):
            value = value.dropna(subset=['mb_sn']).drop_duplicates(subset=['mb_sn'])
            if data is None:
                data = value
            else:
                # 중복 열 유지
                data = pd.merge(data, value, on='mb_sn', how='outer', suffixes=(None, f'_qpoll_{idx}'))

# 병합 후 생성된 접미사 리스트 동적으로 추출
def get_suffixes(data, columns):
    suffixes = set()
    for col in columns:
        suffixes.update([c.replace(col, '') for c in data.columns if c.startswith(col)])
    return list(suffixes)

# 일관되지 않은 응답을 찾는 함수
def find_inconsistent_responses_with_suffix(data, columns):
    suffixes = get_suffixes(data, columns)  # 동적으로 접미사 리스트 생성
    inconsistent_rows = []
    for col in columns:
        # 중복 열 이름 생성
        cols_to_check = [f"{col}{suffix}" for suffix in suffixes if f"{col}{suffix}" in data.columns]
        print(f"Checking column: {col}, Columns to check: {cols_to_check}")  # 어떤 열을 검사 중인지 출력
        if len(cols_to_check) > 1:
            # 동일한 mb_sn에 대해 값이 일관되지 않은 경우 확인
            inconsistent = data.groupby('mb_sn')[cols_to_check].apply(lambda x: x.nunique() > 1)
            inconsistent_mb_sn = inconsistent.any(axis=1)
            print(f"Inconsistent mb_sn for column {col}: {inconsistent_mb_sn[inconsistent_mb_sn].index.tolist()}")  # 중복된 mb_sn 출력
            inconsistent_rows.extend(inconsistent_mb_sn[inconsistent_mb_sn].index.tolist())
    
    # 중복된 mb_sn 제거
    inconsistent_mb_sn = set(inconsistent_rows)
    print(f"Final inconsistent mb_sn: {inconsistent_mb_sn}")  # 최종 중복된 mb_sn 출력
    return data[data['mb_sn'].isin(inconsistent_mb_sn)]

# 확인할 컬럼 리스트
columns_to_check = ['Carrier', 'Sex', 'Birth', 'Region']

# 일관되지 않은 응답 찾기
inconsistent_data = find_inconsistent_responses_with_suffix(data, columns_to_check)

# 결과 출력
print("Inconsistent data:")
print(inconsistent_data)

Checking column: Carrier, Columns to check: ['Carrier', 'Carrier_qpoll_25', 'Carrier_qpoll_16', 'Carrier_qpoll_24', 'Carrier_qpoll_14', 'Carrier_qpoll_27', 'Carrier_qpoll_17', 'Carrier_qpoll_31', 'Carrier_qpoll_28', 'Carrier_qpoll_4', 'Carrier_qpoll_8', 'Carrier_qpoll_33', 'Carrier_qpoll_1', 'Carrier_qpoll_29', 'Carrier_qpoll_22', 'Carrier_qpoll_6', 'Carrier_qpoll_26', 'Carrier_qpoll_11', 'Carrier_qpoll_3', 'Carrier_qpoll_12', 'Carrier_qpoll_21', 'Carrier_qpoll_7', 'Carrier_qpoll_5', 'Carrier_qpoll_19', 'Carrier_qpoll_9', 'Carrier_qpoll_32', 'Carrier_qpoll_23', 'Carrier_qpoll_15', 'Carrier_qpoll_2', 'Carrier_qpoll_10', 'Carrier_qpoll_18', 'Carrier_qpoll_30', 'Carrier_qpoll_34', 'Carrier_qpoll_13', 'Carrier_qpoll_20']
Inconsistent mb_sn for column Carrier: []
Checking column: Sex, Columns to check: ['Sex', 'Sex_qpoll_25', 'Sex_qpoll_16', 'Sex_qpoll_24', 'Sex_qpoll_14', 'Sex_qpoll_27', 'Sex_qpoll_17', 'Sex_qpoll_31', 'Sex_qpoll_28', 'Sex_qpoll_4', 'Sex_qpoll_8', 'Sex_qpoll_33', 'Sex_qpol

In [52]:
for col in ['Carrier', 'Sex', 'Birth', 'Region', '설문일시']:
    cols_to_fill = [c for c in data.columns if c.startswith(col)]
    data[col] = data[cols_to_fill].bfill(axis=1)[col]

data = data.loc[:, ~data.columns.str.contains('_qpoll_')]

welcome1['data'] = welcome1['data'].dropna(subset=['mb_sn']).drop_duplicates(subset=['mb_sn'])
welcome2['data'] = welcome2['data'].dropna(subset=['mb_sn']).drop_duplicates(subset=['mb_sn'])

data = pd.merge(welcome1['data'], data, on='mb_sn', how='outer')
data = pd.merge(welcome2['data'], data, on='mb_sn', how='outer')

C:\Users\india\AppData\Local\Temp\ipykernel_29384\739595865.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[cols_to_fill].bfill(axis=1)[col]
C:\Users\india\AppData\Local\Temp\ipykernel_29384\739595865.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[cols_to_fill].bfill(axis=1)[col]


In [53]:
from datetime import datetime

current_year = datetime.now().year

def format_birth_year(year):
    if pd.notna(year) and isinstance(year, int):
        age = current_year - year
        return f"{year}년 00월 00일 (만 {age} 세)"
    return None

In [54]:
# 성별 결측치 채우기
data['Sex'] = data['Sex'].replace({'남': 'M', '여': 'F'})
data['Q1'] = data['Q1'].fillna(data['Sex'])
# 성별 이상값 None으로 채우기
data.loc[(data['Sex'] != data['Q1']) & data['Q1'].notna() & data['Sex'].notna(), 'Q1'] = None

# 생년 정보 포맷팅 및 나이 재계산 
data['Birth'] = data['Birth'].fillna(data['Q2'].apply(format_birth_year))

# 지역 결측치 채우기
data['Q3_1'] = data['Q3_1'].fillna(data['Region'])

# 컬럼 삭제 및 수정
data.drop(columns=['Sex', 'Q2', 'Region', '설문일시'], inplace=True)
data = data.rename(columns={'Birth': 'Q2', 'Carrier' : 'Q0'})

# 첫번째 레코드 삭제
data = data.iloc[1:].reset_index(drop=True)

# 컬럼 정렬
data = data[sorted(data.columns)]
cols = [col for col in data.columns if col != 'mb_sn']
data = data[['mb_sn'] + cols]

# 데이터 저장
data.to_csv('data.csv', index=False, encoding='utf-8-sig')